<span style="color:lightgreen">

# L4.1 Transcripción de portugués COVOST2

Ya que en este notebook no se realizan traducciones, no tiene cabida reporta COMET y BLEU

<span>

# ASR baseline experiment using Whisper and Covost2 (Spanish-English setup)

In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for ASR on the [Covost2](https://huggingface.co/datasets/facebook/covost2) speech translation corpus (using the Spanish-English setup).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing Word Error Rate, WER)

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
import whisper
from whisper.normalizers.basic import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

import jiwer

model = whisper.load_model("base")

<p style="page-break-after:always;"></p>

Load Covost2 dataset (Spanish-English setup) from Hugging Face. Previously, audio data in the source language (version 4) must be downloaded from [Common Voice](https://commonvoice.mozilla.org/en/datasets)

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("facebook/covost2", CONFIG.trans_code, data_dir=CONFIG.corpus_path)

print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 9158
    })
    validation: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 3318
    })
    test: Dataset({
        features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
        num_rows: 4023
    })
})


Let's take a closer look at the features of the dataset:

In [4]:
raw_datasets["train"].features

{'client_id': Value(dtype='string', id=None),
 'file': Value(dtype='string', id=None),
 'audio': Audio(sampling_rate=16000, mono=True, decode=True, id=None),
 'sentence': Value(dtype='string', id=None),
 'translation': Value(dtype='string', id=None),
 'id': Value(dtype='string', id=None)}

In [5]:
raw_datasets["train"][:5]["file"]

['/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432742.mp3',
 '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432745.mp3',
 '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432746.mp3',
 '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432747.mp3',
 '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432753.mp3']

In [6]:
raw_datasets["train"][:5]["audio"]

[{'path': '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432742.mp3',
  'array': array([2.27373675e-12, 5.45696821e-12, 9.09494702e-13, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00]),
  'sampling_rate': 16000},
 {'path': '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432745.mp3',
  'array': array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
         1.03690354e-07, 1.45667514e-07, 1.11559110e-07]),
  'sampling_rate': 16000},
 {'path': '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432746.mp3',
  'array': array([0., 0., 0., ..., 0., 0., 0.]),
  'sampling_rate': 16000},
 {'path': '/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/data/cv-corpus-24.0-2025-12-05/pt/clips/common_voice_pt_19432747.mp3',
  'array': array([

Show the first 5 Spanish references

In [7]:
raw_datasets["train"][:5]["sentence"]

['Sua alma deve ser muito primitiva para entender essas coisas.',
 'A tristeza em casa é um desejo mórbido de voltar para casa.',
 'O nome original em inglês deste livro não é tão sensacional.',
 'Não? não, obrigado? apenas procurando.',
 'Essa palavra é uma sigla?']

Show the first 5 english translations

In [8]:
raw_datasets["train"][:5]["translation"]

['YourHis/Her soul must be too primitive to understand these things.',
 'Homesickness is a morbid desire to return home.',
 'The original name in English of this book is not so impressive.',
 'No, no, thanks, just looking.',
 'Is this word an acronym?']

<p style="page-break-after:always;"></p>

We pick up the first 1000 audio samples from the training split to be automatically transcribed

In [9]:
data=raw_datasets["validation"][:1000]

Transcribe all the audio data using the Whisper (base) multilingual model. The ASR output is stored in hypotheses.

In [10]:
hypotheses = []
for sample in tqdm(data["file"]):
    hypotheses.append((model.transcribe(sample, language=CONFIG.src_name))['text'])

  0%|          | 0/1000 [00:00<?, ?it/s]

We add the output transcriptions to the data dictionary

In [11]:
data["hypothesis"]=hypotheses

Show the first 5 output transcriptions

In [12]:
data["hypothesis"][:5]

[' Tem que compreender esse fato.',
 ' A três homens asiáticos correm segurando as mãos em uma pista.',
 ' É necessário que a pacitação para rir usar esse trabalho.',
 ' O porco de pinho correu para casa.',
 ' O vestido era a Zonturquesa.']

Transcription hypotheses, references and translations are normalized using the Whisper basic text standardisation/normalization module

In [13]:
normalizer = BasicTextNormalizer()

data["hypothesis_clean"] = [normalizer(text) for text in data["hypothesis"]]
data["sentence_clean"] = [normalizer(text) for text in data["sentence"]]
data["translation_clean"] = [normalizer(text) for text in data["translation"]]

<p style="page-break-after:always;"></p>

Finally, we compute the transcription WER using [JIWER](https://openai.com/index/whisper/) which is a simple and fast python package to evaluate ASR performance.

In [14]:

wer = jiwer.wer(list(data["sentence_clean"]), list(data["hypothesis_clean"]))

print(f"WER: {wer * 100:.2f} %")

WER: 21.37 %


Hypotheses and translations are stored into a Pandas dataframe

In [15]:
dataframe = pd.DataFrame(dict(transcription=data["hypothesis"], sentence=data["sentence"], translation=data["translation"], transcription_clean=data["hypothesis_clean"],  sentence_clean=data["sentence_clean"], translation_clean=data["translation_clean"] ))
pd.set_option('display.max_colwidth', None)
dataframe.head(1)

,transcription,sentence,translation,transcription_clean,sentence_clean,translation_clean
0,Tem que compreender esse fato.,Temos que compreender esse fato.,We need to understand this fact.,tem que compreender esse fato,temos que compreender esse fato,we need to understand this fact


All the data is stored into a file using 'csv' format

In [16]:
dataframe.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'L4.1_ASR_Whisper_Baseline_dev_Covost2.csv'), encoding='utf-8')

# Exercise

Perform a similar experiment using a different Covost2 source-english setup. Evaluate the performance of different whisper models 